#RL Actor Critic PPO Policy/Agent for Motion Planning end to end in MetaDrive Simulator/Environment

You will:
- Formulate autonomous driving as an MDP
- Train a PPO agent (Stable-Baselines version)
- Evaluate generalization
- Deploy and visualize the policy
- Launch a Gradio demo


In [2]:
import os
import sys
import pathlib

# Method 1: Use the notebook's actual path
try:
    # Get the directory where this notebook is running
    notebook_path = os.getcwd()
    project_root = notebook_path
    
    # If we're inside notebooks/, go up one level
    if 'notebooks' in notebook_path:
        project_root = os.path.dirname(notebook_path)
    
except:
    # Fallback: assume current dir is project root
    project_root = os.getcwd()

# Add src to Python path
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Verify the path exists
if not os.path.exists(src_path):
    print(f"❌ ERROR: src path does not exist: {src_path}")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Files in current dir: {os.listdir('.')}")
else:
    print(f"✅ Src path found: {src_path}")
    print(f"Files in src/: {os.listdir(src_path)}")

# Now import
from env_config import get_env_config
from train import create_ppo_model, train_model
from evaluate import evaluate_policy, run_benchmark
from visualize import generate_gif, generate_demo_suite, create_gradio_demo

from stable_baselines3 import PPO
import matplotlib.pyplot as plt

print("\n✅ All modules imported successfully!")


✅ Src path found: /home/vijay2998/meta-drive-ppo-highway-planner/src
Files in src/: ['env_config.py', 'train.py', 'evaluate.py', 'visualize.py', '__init__.py', '__pycache__']


ModuleNotFoundError: No module named 'metadrive'

In [ ]:
!pip install -q git+https://github.com/metadriverse/metadrive.git
!pip install -q stable-baselines3 gymnasium torch gradio imageio

In [ ]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'

In [ ]:
import numpy as np
import torch
import imageio
import gradio as gr

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from metadrive import MetaDriveEnv
from metadrive.engine.engine_utils import close_engine

close_engine()

In [ ]:
ENV_CONFIG = {
    "use_render": False,
    "num_scenarios": 50,
    "start_seed": 0,
    "traffic_density": 0.1,
    "random_lane_width": True,
    "random_agent_model": True,
    "num_agents": 1,
    "horizon": 1000,

    # 🔴 CRITICAL: freeze sensors
    "sensors": {
        "lidar": (),
        "side_detector": (),
        "lane_line_detector": ()
    }
}


## 1. MetaDrive Environment (Single Agent, Safe Setup)

In [ ]:
close_engine()

env = MetaDriveEnv(ENV_CONFIG)
env = Monitor(env)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)

## 2. PPO Actor–Critic Configuration

In [ ]:
policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
    activation_fn=torch.nn.ReLU
)

model = PPO(
    'MlpPolicy',
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
    verbose=1,
)

## 3. Train PPO Agent

In [ ]:
from src.train import create_ppo_model, train_model

env_config = get_env_config("train")
model, env = create_ppo_model(env_config, verbose=1)
model = train_model(model, total_timesteps=100000, save_path="models/ppo_metadrive_safe")

## 4. Evaluation on Unseen Maps

In [ ]:
from src.evaluate import evaluate_policy, run_benchmark

# Evaluate on unseen seeds
mean_reward, std_reward = evaluate_policy(model, config_type="eval", episodes=5)
print(f"Final Evaluation: {mean_reward:.2f} ± {std_reward:.2f}")

# Optional: Run benchmark vs random agent
benchmark = run_benchmark(model)
print(f"Benchmark - PPO: {benchmark['ppo_mean']:.2f}, Random: {benchmark['random_mean']:.2f}")


## 5. Policy Deployment & Visualization

In [ ]:
close_engine()

render_env = MetaDriveEnv(ENV_CONFIG)

obs, _ = render_env.reset()
frames = []

for _ in range(800):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, _ = render_env.step(action)
    frame = render_env.render(mode='top_down', screen_size=(500, 500))
    frames.append(frame)
    if term or trunc:
        break

render_env.close()
imageio.mimsave('ppo_safe_demo.gif', frames, fps=10)
print('Saved ppo_safe_demo.gif')

In [ ]:
from src.visualize import generate_demo_suite, create_gradio_demo

# Generate all 9 GIFs automatically
gif_index = generate_demo_suite(model, output_dir="outputs/demo")

# Optional: Launch Gradio demo
# demo = create_gradio_demo(gif_index)
# demo.launch(share=True)


## 6. Gradio Interactive Demo

In [ ]:
import gradio as gr

def show_gif(seed, traffic_density):
    key = (int(seed), float(traffic_density))
    return GIF_INDEX.get(key)


In [ ]:

demo = gr.Interface(
    fn=show_gif,
    inputs=[
        gr.Dropdown(
            choices=[1008, 1010, 1012],
            label="Scenario Seed",
            value=1000
        ),
        gr.Dropdown(
            choices=[0.05, 0.1, 0.2],
            label="Traffic Density",
            value=0.1
        ),
    ],
    outputs=gr.Image(type="filepath"),
    title="PPO MetaDrive Results Viewer",
    description="""
    This demo displays pre-generated PPO rollouts.
    MetaDrive runs offline; Gradio only visualizes results.
    """
)

demo.launch(share=True, debug=True)
